### 1: Imports

In [1]:
%pip list

Package                 Version
----------------------- -----------
asttokens               3.0.1
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.20
decorator               5.2.1
executing               2.2.1
filelock                3.25.2
fonttools               4.62.1
fsspec                  2026.3.0
ipykernel               7.2.0
ipython                 9.12.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
Jinja2                  3.1.6
joblib                  1.5.3
jupyter_client          8.8.0
jupyter_core            5.9.1
kiwisolver              1.5.0
MarkupSafe              3.0.3
matplotlib              3.10.8
matplotlib-inline       0.2.1
mpmath                  1.3.0
nest-asyncio            1.6.0
networkx                3.6.1
numpy                   2.4.3
packaging               26.0
pandas                  3.0.1
parso                   0.8.6
pillow                 

In [2]:
import xgboost
print(xgboost.__version__)
print(xgboost.__file__)

1.7.6
c:\Users\ezsha\StudioProjects\pain_tracker\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('imports ok')

imports ok


### 2: Load Data

In [4]:
data_path = os.path.join(BASE_DIR, 'processed_data.csv')
df = pd.read_csv(data_path)

feature_cols = [col for col in df.columns if col not in ['pain_level', 'person_id']]
input_size = len(feature_cols)
people_ids = df['person_id'].unique()

print(f'Total windows: {len(df)}')
print(f'People: {people_ids}')
print(f'Features: {input_size}')
print(f'\nClass balance:')
print(df.groupby(['person_id', 'pain_level']).size().unstack(fill_value=0))

# split person 0's data 80/20 for demo-focused evaluation
person0_df = df[df['person_id'] == 0]
others_df  = df[df['person_id'] != 0]

p0_train, p0_test = train_test_split(
    person0_df, test_size=0.2, random_state=42, stratify=person0_df['pain_level']
)

# training pool: everyone else + 80% of person 0
train_pool = pd.concat([others_df, p0_train], ignore_index=True)
test_pool  = p0_test

X_train_p0 = train_pool[feature_cols].values
y_train_p0 = train_pool['pain_level'].values.astype(int)
X_test_p0  = test_pool[feature_cols].values
y_test_p0  = test_pool['pain_level'].values.astype(int)

print(f'\nPerson 0 train samples: {len(p0_train)}')
print(f'Person 0 test samples:  {len(p0_test)}')
print(f'Total training pool:    {len(train_pool)}')

X      = df[feature_cols].values
y      = df['pain_level'].values.astype(int)
groups = df['person_id'].values

Total windows: 1561
People: [0 1 2]
Features: 15

Class balance:
pain_level    0    1    2    3
person_id                     
0           206  139  260  219
1            84  144   84   75
2            83  114   73   80

Person 0 train samples: 659
Person 0 test samples:  165
Total training pool:    1396


### 3: Helper Functions

In [5]:
def print_confusion_matrix(true, pred, title, num_classes=4):
    cm = confusion_matrix(true, pred)
    print(f'\n{title}')
    for i, row in enumerate(cm):
        print(f'true L{i}: {row.tolist()}')

print('helpers defined')

helpers defined


### 4: XGBoost Base Model Evaluation

In [6]:
scaler_xgb = StandardScaler()
X_tr_xgb   = scaler_xgb.fit_transform(X_train_p0)
X_te_xgb   = scaler_xgb.transform(X_test_p0)

weights_xgb = compute_sample_weight('balanced', y_train_p0)
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=1
)
X_tr_xgb_df = pd.DataFrame(X_tr_xgb, columns=feature_cols)
X_te_xgb_df = pd.DataFrame(X_te_xgb, columns=feature_cols)

xgb_model.fit(X_tr_xgb_df, y_train_p0, sample_weight=weights_xgb)
xgb_preds = xgb_model.predict(X_te_xgb_df)

xgb_acc = accuracy_score(y_test_p0, xgb_preds) * 100
print(f'XGBoost Accuracy on Person 0: {xgb_acc:.2f}%')
print_confusion_matrix(y_test_p0, xgb_preds, 'XGBoost Confusion Matrix')

XGBoost Accuracy on Person 0: 92.73%

XGBoost Confusion Matrix
true L0: [40, 0, 0, 1]
true L1: [0, 22, 5, 1]
true L2: [0, 1, 49, 2]
true L3: [1, 0, 1, 42]


In [7]:
"""
import pandas as pd

# get feature importance from the trained model
importance = xgb_model.get_booster().get_score(importance_type='gain')

# gain = how much each feature improves accuracy when it's used for a split
# higher = more important
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v} 
    for k, v in importance.items()
]).sort_values('importance', ascending=False)

print('Feature Importance (by gain):')
print(importance_df.to_string(index=False))
"""

"\nimport pandas as pd\n\n# get feature importance from the trained model\nimportance = xgb_model.get_booster().get_score(importance_type='gain')\n\n# gain = how much each feature improves accuracy when it's used for a split\n# higher = more important\nimportance_df = pd.DataFrame([\n    {'feature': k, 'importance': v} \n    for k, v in importance.items()\n]).sort_values('importance', ascending=False)\n\nprint('Feature Importance (by gain):')\nprint(importance_df.to_string(index=False))\n"

### 5: XGBoost Personalization (Fine-Tuning)

In [ ]:
# simulate the real product flow:
# 1. base model already trained on all people
# 2. new user (person 0) logs pain naturally over time
# 3. once enough events accumulate, warm-start fine-tune on their data

X_p0_natural = pd.DataFrame(scaler_xgb.transform(p0_train[feature_cols].values), columns=feature_cols)
y_p0_natural = p0_train["pain_level"].values.astype(int)

personalized_model = XGBClassifier(
    n_estimators=20,
    max_depth=3,
    learning_rate=0.05,
    eval_metric="mlogloss",
    verbosity=0,
    n_jobs=1
)
personalized_model.fit(
    X_p0_natural, y_p0_natural,
    xgb_model=xgb_model.get_booster() 
)

personalized_preds = personalized_model.predict(X_te_xgb)
personalized_acc = accuracy_score(y_test_p0, personalized_preds) * 100

print(f"Base XGBoost accuracy on Person 0:         {xgb_acc:.2f}%")
print(f"Personalized XGBoost accuracy on Person 0:  {personalized_acc:.2f}%")
print(f"Improvement: {personalized_acc - xgb_acc:+.2f}%")
print_confusion_matrix(y_test_p0, personalized_preds, "Personalized XGBoost Confusion Matrix")

#personalized_model.save_model(os.path.join(BASE_DIR, "personalized_model_p0.json"))
personalized_model.get_booster().save_model(os.path.join(BASE_DIR, "personalized_model_p0.json"))
print("\npersonalized_model_p0.json saved.")
print("In the real app, this file would live on the user phone and update as they log more data.")

Base XGBoost accuracy on Person 0:         92.73%
Personalized XGBoost accuracy on Person 0:  95.76%
Improvement: +3.03%

Personalized XGBoost Confusion Matrix
true L0: [40, 0, 0, 1]
true L1: [0, 24, 4, 0]
true L2: [0, 0, 52, 0]
true L3: [1, 0, 1, 42]


TypeError: `_estimator_type` undefined.  Please use appropriate mixin to define estimator type.

### 6: Train Final Model & Export

In [9]:
sample_weights_final = np.where(df['person_id'].values == 0, 3.0, 1.0)

final_scaler = StandardScaler()
X_all = final_scaler.fit_transform(X)

final_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=1
)
final_model.fit(X_all, y, sample_weight=sample_weights_final)

# Save XGBoost Model
#final_model.save_model(os.path.join(BASE_DIR, 'base_model.bin'))
raw = final_model.get_booster().save_raw()
with open(os.path.join(BASE_DIR, 'base_model.bin'), 'wb') as f:
    f.write(raw)

# Save Python Scaler
joblib.dump(final_scaler, os.path.join(BASE_DIR, 'scaler.pkl'))

# Export Scaler Parameters for Android (Direct Inference)
scaler_params = {
    "mean": final_scaler.mean_.tolist(),
    "scale": final_scaler.scale_.tolist()
}
with open(os.path.join(BASE_DIR, 'scaler_params.json'), 'w') as f:
    json.dump(scaler_params, f)

print('saved: base_model.bin (XGBoost tree data)')
print('saved: scaler.pkl (StandardScaler for Python scripts)')
print('saved: scaler_params.json (Mathematical parameters for Kotlin/Android)')

saved: base_model.bin (XGBoost tree data)
saved: scaler.pkl (StandardScaler for Python scripts)
saved: scaler_params.json (Mathematical parameters for Kotlin/Android)


### Testing Setup

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
from sklearn.metrics import accuracy_score, confusion_matrix

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))

def evaluate_raw_dataset(raw_csv_path):
    print(f"--- Evaluating Raw Dataset: {os.path.basename(raw_csv_path)} ---")
    
    # 1. Load Model and Scaler
    model = xgb.XGBClassifier()
    model.load_model(os.path.join(BASE_DIR, 'base_model.json'))
    scaler = joblib.load(os.path.join(BASE_DIR, 'scaler.pkl'))
    
    # 2. Load Raw Data
    df = pd.read_csv(raw_csv_path)
    
    # Drop artifacts and baselines (as done in preprocess.py)
    df = df[(df['pain_level'] != -1) & (df['hr'] != 0)].copy()
    if df.empty:
        print("Dataset is empty after removing artifacts.")
        return

    # 3. Parse Pipe Strings
    def parse_pipe_string(val, cast_type):
        if pd.isna(val) or str(val).strip() in ('', 'nan', 'None'): return []
        return [cast_type(v) for v in str(val).strip().split('|') if v.strip()]

    df['ibi'] = df['ibi'].apply(lambda x: parse_pipe_string(x, int))
    df['ecg'] = df['ecg'].apply(lambda x: parse_pipe_string(x, float))
    
    # Handle missing IBI
    df['ibi'] = df['ibi'].apply(lambda x: np.nan if len(x) == 0 else x)
    df['ibi'] = df['ibi'].ffill().bfill()

    # 4. Extract Features (Sliding Window of 10)
    WINDOW_SIZE = 10
    STEP_SIZE = 1
    extracted_windows = []
    
    # Group by contiguous pain levels to avoid bleeding data across pain events
    df['event_id'] = (df['pain_level'] != df['pain_level'].shift()).cumsum()
    
    for _, event_df in df.groupby('event_id'):
        event_df = event_df.reset_index(drop=True)
        if len(event_df) < WINDOW_SIZE: continue
            
        pain_level = event_df['pain_level'].iloc[0]
        
        for start in range(0, len(event_df) - WINDOW_SIZE + 1, STEP_SIZE):
            window = event_df.iloc[start:start + WINDOW_SIZE]
            
            # HR metrics
            hr = window['hr'].values
            
            # IBI metrics
            all_ibis = [val for sublist in window['ibi'] for val in sublist]
            ibis_arr = np.array(all_ibis) if len(all_ibis) > 1 else np.array([0])
            first_diffs = np.abs(np.diff(ibis_arr))
            
            # ECG metric
            first_ecg = window['ecg'].iloc[0]
            
            features = {
                'hr_mean': np.mean(hr),
                'hr_std': np.std(hr),
                'hr_min': np.min(hr),
                'hr_max': np.max(hr),
                'hr_range': np.max(hr) - np.min(hr),
                
                'ibi_mean': np.mean(ibis_arr),
                'ibi_std': np.std(ibis_arr),
                'ibi_min': np.min(ibis_arr),
                'ibi_max': np.max(ibis_arr),
                'ibi_range': np.max(ibis_arr) - np.min(ibis_arr),
                'ibi_first_diff_mean': np.mean(first_diffs) if len(first_diffs) > 0 else 0,
                'ibi_second_diff_mean': np.mean(np.abs(np.diff(first_diffs))) if len(first_diffs) > 1 else 0,
                'rmssd': np.sqrt(np.mean(first_diffs**2)) if len(first_diffs) > 0 else 0,
                'pnn50': np.sum(first_diffs > 50) / len(ibis_arr) if len(ibis_arr) > 0 else 0,
                
                'ecg_var': np.var(first_ecg) if len(first_ecg) > 0 else 0.0,
                
                'actual_pain': pain_level
            }
            extracted_windows.append(features)

    features_df = pd.DataFrame(extracted_windows).fillna(0)
    
    if features_df.empty:
        print("Not enough data to form a 10-second window.")
        return

    # 5. Separate features from labels
    feature_cols = [col for col in features_df.columns if col != 'actual_pain']
    X_raw = features_df[feature_cols].values
    y_true = features_df['actual_pain'].values.astype(int)

    # 6. Scale and Predict
    X_scaled = scaler.transform(X_raw)
    y_pred = model.predict(X_scaled)

    # 7. Print Results
    acc = accuracy_score(y_true, y_pred) * 100
    print(f"Total valid 10-second windows tested: {len(X_raw)}")
    print(f"Model Accuracy on this dataset: {acc:.2f}%")
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    for i, row in enumerate(cm):
        print(f"Actual L{i}: {row.tolist()}")

# --- RUN THE TEST ---
# Replace this path with the path to the specific raw dataset you want to test
test_file_path = os.path.join(BASE_DIR, '..', 'pain_tracker_data', 'test_files', 'ethan_pain_tracker_20260329_015001.csv')

# Only run if the file exists
if os.path.exists(test_file_path):
    evaluate_raw_dataset(test_file_path)
else:
    print(f"Could not find test file at: {test_file_path}")

Could not find test file at: c:\Users\ezsha\StudioProjects\pain_tracker\ML\..\pain_tracker_data\test_files\ethan_pain_tracker_20260329_015001.csv
